In [ ]:
#| echo: false
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

from IPython.display import Markdown, display


def _lib_dir() -> Path:
    """Locate the katapult package directory (contains pyproject.toml and tests/)."""
    if "QUARTO_PROJECT_DIR" in os.environ:
        d = Path(os.environ["QUARTO_PROJECT_DIR"]).resolve() / "lib"
        if (d / "pyproject.toml").is_file():
            return d
    here = Path.cwd().resolve()
    for ancestor in [here, *here.parents]:
        d = ancestor / "lib"
        if (d / "pyproject.toml").is_file():
            return d
    raise FileNotFoundError(
        "Could not find lib/pyproject.toml. Run Quarto from the repo root or open the notebook from the katapult project."
    )


def run_pytest(*extra_args: str) -> subprocess.CompletedProcess[str]:
    """Run pytest in lib/ and print stdout/stderr for display in the notebook."""
    lib = _lib_dir()
    venv_py = lib / ".venv" / "bin" / "python"
    exe = str(venv_py) if venv_py.is_file() else sys.executable
    cmd = [exe, "-m", "pytest", "tests/", "-v", "--tb=short", *extra_args]
    result = subprocess.run(cmd, cwd=str(lib), capture_output=True, text=True)
    out = (result.stdout or "") + (result.stderr or "")
    if not out.strip():
        out = "(no output)"
    display(Markdown(f"```text\n{out.rstrip()}\n```"))
    display(Markdown(f"**Exit code:** `{result.returncode}`"))
    return result

# Katapult library unit tests

This notebook documents the pytest suite under `lib/tests/` and runs each group so you can see live results when you execute the cells (or when Quarto renders this page).

**Source:** [`lib/tests/test_commands.py`](../../lib/tests/test_commands.py) — helpers in [`katapult.commands`](../../lib/src/katapult/commands.py) and the `kat` CLI (`init`, `hub`, `config`, and the internal `rich` demo command).


## Overview

Tests fall into three layers: pure path/JSON helpers used by `kat init`, Click CLI exercises with mocks (no real Docker or cookiecutter prompts), and optional grouping by pytest `-k` expression.

```mermaid
flowchart TB
  subgraph helpers [Helper functions]
    merge["_merge_overrides"]
    cwr["_apply_copy_without_render"]
    otc["_override_template_has_content"]
  end
  subgraph cli [CLI commands]
    init["init"]
    hub["hub"]
    cfg["config"]
    rich["rich"]
  end
  merge --> init
  cwr --> init
  otc --> init
```


## `_merge_overrides`

When `kat init` applies `~/.katapult/template/` overrides, files are copied into the cookiecutter project directory inside a temporary merged template, preserving relative paths.

- **`test_merge_overrides_copies_nested_files`** — nested directories and files land under `dst` with the same structure.
- **`test_merge_overrides_skips_directories`** — empty directories under the source do not create files under `dst` (only files are copied).

```mermaid
flowchart LR
  src["override src"] --> merge["_merge_overrides"]
  merge --> dst["dst = .../ project_slug /"]
```


In [2]:
run_pytest("-k", "merge_overrides")

```text
[1m============================= test session starts ==============================[0m
platform linux -- Python 3.12.3, pytest-9.0.2, pluggy-1.6.0 -- /content/lib/.venv/bin/python
cachedir: .pytest_cache
rootdir: /content/lib
configfile: pyproject.toml
[1mcollecting ... [0mcollected 23 items / 21 deselected / 2 selected

tests/test_commands.py::test_merge_overrides_copies_nested_files [32mPASSED[0m[32m  [ 50%][0m
tests/test_commands.py::test_merge_overrides_skips_directories [32mPASSED[0m[32m    [100%][0m

[32m======================= [32m[1m2 passed[0m, [33m21 deselected[0m[32m in 0.39s[0m[32m =======================[0m
```

**Exit code:** `0`

CompletedProcess(args=['/content/lib/.venv/bin/python', '-m', 'pytest', 'tests/', '-v', '--tb=short', '-k', 'merge_overrides'], returncode=0, stdout='\x1b============================= test session starts ==============================\x1b\nplatform linux -- Python 3.12.3, pytest-9.0.2, pluggy-1.6.0 -- /content/lib/.venv/bin/python\ncachedir: .pytest_cache\nrootdir: /content/lib\nconfigfile: pyproject.toml\n\x1bcollecting ... \x1bcollected 23 items / 21 deselected / 2 selected\n\ntests/test_commands.py::test_merge_overrides_copies_nested_files \x1bPASSED\x1b\x1b  [ 50%]\x1b\ntests/test_commands.py::test_merge_overrides_skips_directories \x1bPASSED\x1b\x1b    [100%]\x1b\n\n\x1b======================= \x1b\x1b2 passed\x1b, \x1b21 deselected\x1b\x1b in 0.39s\x1b\x1b =======================\x1b\n', stderr='')

## `_apply_copy_without_render`

Merges glob patterns from `~/.katapult/copy_without_render` into the template's `cookiecutter.json` field `_copy_without_render` so those paths are copied without Jinja templating.

| Test | What it checks |
|------|----------------|
| `merges_patterns` | Lines from the ignore file append to existing list; `#` comments and blanks skipped |
| `deduplicates` | Duplicate patterns appear once |
| `non_list_existing_reset` | If existing value is not a list, it is replaced |
| `missing_ignore_noop` | No ignore file → no change to JSON |
| `empty_patterns_noop` | Only comments/empty lines → no change |
| `invalid_json_raises` | Bad `cookiecutter.json` → `JSONDecodeError` |
| `missing_cookiecutter_raises` | Missing JSON path → `FileNotFoundError` |


In [3]:
run_pytest("-k", "apply_copy_without_render")

```text
[1m============================= test session starts ==============================[0m
platform linux -- Python 3.12.3, pytest-9.0.2, pluggy-1.6.0 -- /content/lib/.venv/bin/python
cachedir: .pytest_cache
rootdir: /content/lib
configfile: pyproject.toml
[1mcollecting ... [0mcollected 23 items / 16 deselected / 7 selected

tests/test_commands.py::test_apply_copy_without_render_merges_patterns [32mPASSED[0m[32m [ 14%][0m
tests/test_commands.py::test_apply_copy_without_render_deduplicates [32mPASSED[0m[32m [ 28%][0m
tests/test_commands.py::test_apply_copy_without_render_non_list_existing_reset [32mPASSED[0m[32m [ 42%][0m
tests/test_commands.py::test_apply_copy_without_render_missing_ignore_noop [32mPASSED[0m[32m [ 57%][0m
tests/test_commands.py::test_apply_copy_without_render_empty_patterns_noop [32mPASSED[0m[32m [ 71%][0m
tests/test_commands.py::test_apply_copy_without_render_invalid_json_raises [32mPASSED[0m[32m [ 85%][0m
tests/test_commands.py::test_apply_copy_without_render_missing_cookiecutter_raises [32mPASSED[0m[32m [100%][0m

[32m======================= [32m[1m7 passed[0m, [33m16 deselected[0m[32m in 0.39s[0m[32m =======================[0m
```

**Exit code:** `0`

CompletedProcess(args=['/content/lib/.venv/bin/python', '-m', 'pytest', 'tests/', '-v', '--tb=short', '-k', 'apply_copy_without_render'], returncode=0, stdout='\x1b============================= test session starts ==============================\x1b\nplatform linux -- Python 3.12.3, pytest-9.0.2, pluggy-1.6.0 -- /content/lib/.venv/bin/python\ncachedir: .pytest_cache\nrootdir: /content/lib\nconfigfile: pyproject.toml\n\x1bcollecting ... \x1bcollected 23 items / 16 deselected / 7 selected\n\ntests/test_commands.py::test_apply_copy_without_render_merges_patterns \x1bPASSED\x1b\x1b [ 14%]\x1b\ntests/test_commands.py::test_apply_copy_without_render_deduplicates \x1bPASSED\x1b\x1b [ 28%]\x1b\ntests/test_commands.py::test_apply_copy_without_render_non_list_existing_reset \x1bPASSED\x1b\x1b [ 42%]\x1b\ntests/test_commands.py::test_apply_copy_without_render_missing_ignore_noop \x1bPASSED\x1b\x1b [ 57%]\x1b\ntests/test_commands.py::test_apply_copy_without_render_empty_patterns_noop \x1bPASSED\x1b

## `_override_template_has_content`

Decides whether `kat init` should use the merge-and-temp-template path: **True** only if the override directory exists and contains at least one file (any depth).

- Missing path → False  
- Empty directory → False  
- File at root or nested → True


In [4]:
run_pytest("-k", "override_template_has_content")

```text
[1m============================= test session starts ==============================[0m
platform linux -- Python 3.12.3, pytest-9.0.2, pluggy-1.6.0 -- /content/lib/.venv/bin/python
cachedir: .pytest_cache
rootdir: /content/lib
configfile: pyproject.toml
[1mcollecting ... [0mcollected 23 items / 19 deselected / 4 selected

tests/test_commands.py::test_override_template_has_content_false_when_missing [32mPASSED[0m[32m [ 25%][0m
tests/test_commands.py::test_override_template_has_content_false_when_empty_dir [32mPASSED[0m[32m [ 50%][0m
tests/test_commands.py::test_override_template_has_content_true_with_file [32mPASSED[0m[32m [ 75%][0m
tests/test_commands.py::test_override_template_has_content_true_nested_file [32mPASSED[0m[32m [100%][0m

[32m======================= [32m[1m4 passed[0m, [33m19 deselected[0m[32m in 0.38s[0m[32m =======================[0m
```

**Exit code:** `0`

CompletedProcess(args=['/content/lib/.venv/bin/python', '-m', 'pytest', 'tests/', '-v', '--tb=short', '-k', 'override_template_has_content'], returncode=0, stdout='\x1b============================= test session starts ==============================\x1b\nplatform linux -- Python 3.12.3, pytest-9.0.2, pluggy-1.6.0 -- /content/lib/.venv/bin/python\ncachedir: .pytest_cache\nrootdir: /content/lib\nconfigfile: pyproject.toml\n\x1bcollecting ... \x1bcollected 23 items / 19 deselected / 4 selected\n\ntests/test_commands.py::test_override_template_has_content_false_when_missing \x1bPASSED\x1b\x1b [ 25%]\x1b\ntests/test_commands.py::test_override_template_has_content_false_when_empty_dir \x1bPASSED\x1b\x1b [ 50%]\x1b\ntests/test_commands.py::test_override_template_has_content_true_with_file \x1bPASSED\x1b\x1b [ 75%]\x1b\ntests/test_commands.py::test_override_template_has_content_true_nested_file \x1bPASSED\x1b\x1b [100%]\x1b\n\n\x1b======================= \x1b\x1b4 passed\x1b, \x1b19 deselected\

## CLI: `rich` and `config`

- **`rich`** — demo command (not registered on the main `kat` group); prints a Rich table; exit code 0.
- **`config`** — appends PATH-augmentation block to `~/.bashrc` once; second run reports already present. Tests patch `Path.home` to a temporary directory.

```mermaid
sequenceDiagram
  participant U as User
  participant K as kat config
  participant B as .bashrc
  U->>K: invoke
  K->>B: append marker block if missing
  K-->>U: message
```


In [5]:
run_pytest("-k", "rich or config")

```text
[1m============================= test session starts ==============================[0m
platform linux -- Python 3.12.3, pytest-9.0.2, pluggy-1.6.0 -- /content/lib/.venv/bin/python
cachedir: .pytest_cache
rootdir: /content/lib
configfile: pyproject.toml
[1mcollecting ... [0mcollected 23 items / 20 deselected / 3 selected

tests/test_commands.py::test_rich_exits_zero [32mPASSED[0m[32m                      [ 33%][0m
tests/test_commands.py::test_config_appends_bashrc_once [32mPASSED[0m[32m           [ 66%][0m
tests/test_commands.py::test_config_creates_bashrc [32mPASSED[0m[32m                [100%][0m

[32m======================= [32m[1m3 passed[0m, [33m20 deselected[0m[32m in 0.41s[0m[32m =======================[0m
```

**Exit code:** `0`

CompletedProcess(args=['/content/lib/.venv/bin/python', '-m', 'pytest', 'tests/', '-v', '--tb=short', '-k', 'rich or config'], returncode=0, stdout='\x1b============================= test session starts ==============================\x1b\nplatform linux -- Python 3.12.3, pytest-9.0.2, pluggy-1.6.0 -- /content/lib/.venv/bin/python\ncachedir: .pytest_cache\nrootdir: /content/lib\nconfigfile: pyproject.toml\n\x1bcollecting ... \x1bcollected 23 items / 20 deselected / 3 selected\n\ntests/test_commands.py::test_rich_exits_zero \x1bPASSED\x1b\x1b                      [ 33%]\x1b\ntests/test_commands.py::test_config_appends_bashrc_once \x1bPASSED\x1b\x1b           [ 66%]\x1b\ntests/test_commands.py::test_config_creates_bashrc \x1bPASSED\x1b\x1b                [100%]\x1b\n\n\x1b======================= \x1b\x1b3 passed\x1b, \x1b20 deselected\x1b\x1b in 0.41s\x1b\x1b =======================\x1b\n', stderr='')

## CLI: `kat init`

Cookiecutter is **mocked** so tests never prompt interactively.

- **`--no-overrides`** — calls `cookiecutter` with the built-in template directory only.
- **With overrides** — copies template to a temp dir, merges `~/.katapult/template/` into `{{cookiecutter.project_slug}}/`, then calls `cookiecutter` on the merged tree. One test asserts merged files inside a `cookiecutter` side-effect while the temp dir still exists.
- **`--no-overrides` with template files present** — still uses the stock template path (skips merge).


In [6]:
run_pytest("-k", "init")

```text
[1m============================= test session starts ==============================[0m
platform linux -- Python 3.12.3, pytest-9.0.2, pluggy-1.6.0 -- /content/lib/.venv/bin/python
cachedir: .pytest_cache
rootdir: /content/lib
configfile: pyproject.toml
[1mcollecting ... [0mcollected 23 items / 20 deselected / 3 selected

tests/test_commands.py::test_init_no_overrides_calls_cookiecutter_template_dir [32mPASSED[0m[32m [ 33%][0m
tests/test_commands.py::test_init_with_overrides_merges_and_calls_cookiecutter [32mPASSED[0m[32m [ 66%][0m
tests/test_commands.py::test_init_no_overrides_skips_merge_even_with_template_files [32mPASSED[0m[32m [100%][0m

[32m======================= [32m[1m3 passed[0m, [33m20 deselected[0m[32m in 0.39s[0m[32m =======================[0m
```

**Exit code:** `0`

CompletedProcess(args=['/content/lib/.venv/bin/python', '-m', 'pytest', 'tests/', '-v', '--tb=short', '-k', 'init'], returncode=0, stdout='\x1b============================= test session starts ==============================\x1b\nplatform linux -- Python 3.12.3, pytest-9.0.2, pluggy-1.6.0 -- /content/lib/.venv/bin/python\ncachedir: .pytest_cache\nrootdir: /content/lib\nconfigfile: pyproject.toml\n\x1bcollecting ... \x1bcollected 23 items / 20 deselected / 3 selected\n\ntests/test_commands.py::test_init_no_overrides_calls_cookiecutter_template_dir \x1bPASSED\x1b\x1b [ 33%]\x1b\ntests/test_commands.py::test_init_with_overrides_merges_and_calls_cookiecutter \x1bPASSED\x1b\x1b [ 66%]\x1b\ntests/test_commands.py::test_init_no_overrides_skips_merge_even_with_template_files \x1bPASSED\x1b\x1b [100%]\x1b\n\n\x1b======================= \x1b\x1b3 passed\x1b, \x1b20 deselected\x1b\x1b in 0.39s\x1b\x1b =======================\x1b\n', stderr='')

## CLI: `kat hub`

Docker is **fully mocked** (`docker.from_env`). Tests drive confirms with `CliRunner` input:

- Traefik already running → no `create` / `run`.
- No network → user confirms → `networks.create` and `containers.run`.
- User declines network or Traefik launch → abort messages, no side effects.


In [7]:
run_pytest("-k", "hub")

```text
[1m============================= test session starts ==============================[0m
platform linux -- Python 3.12.3, pytest-9.0.2, pluggy-1.6.0 -- /content/lib/.venv/bin/python
cachedir: .pytest_cache
rootdir: /content/lib
configfile: pyproject.toml
[1mcollecting ... [0mcollected 23 items / 19 deselected / 4 selected

tests/test_commands.py::test_hub_traefik_already_running [32mPASSED[0m[32m          [ 25%][0m
tests/test_commands.py::test_hub_creates_network_and_launches_traefik [32mPASSED[0m[32m [ 50%][0m
tests/test_commands.py::test_hub_aborts_when_network_declined [32mPASSED[0m[32m     [ 75%][0m
tests/test_commands.py::test_hub_aborts_when_traefik_launch_declined [32mPASSED[0m[32m [100%][0m

[32m======================= [32m[1m4 passed[0m, [33m19 deselected[0m[32m in 0.37s[0m[32m =======================[0m
```

**Exit code:** `0`

CompletedProcess(args=['/content/lib/.venv/bin/python', '-m', 'pytest', 'tests/', '-v', '--tb=short', '-k', 'hub'], returncode=0, stdout='\x1b============================= test session starts ==============================\x1b\nplatform linux -- Python 3.12.3, pytest-9.0.2, pluggy-1.6.0 -- /content/lib/.venv/bin/python\ncachedir: .pytest_cache\nrootdir: /content/lib\nconfigfile: pyproject.toml\n\x1bcollecting ... \x1bcollected 23 items / 19 deselected / 4 selected\n\ntests/test_commands.py::test_hub_traefik_already_running \x1bPASSED\x1b\x1b          [ 25%]\x1b\ntests/test_commands.py::test_hub_creates_network_and_launches_traefik \x1bPASSED\x1b\x1b [ 50%]\x1b\ntests/test_commands.py::test_hub_aborts_when_network_declined \x1bPASSED\x1b\x1b     [ 75%]\x1b\ntests/test_commands.py::test_hub_aborts_when_traefik_launch_declined \x1bPASSED\x1b\x1b [100%]\x1b\n\n\x1b======================= \x1b\x1b4 passed\x1b, \x1b19 deselected\x1b\x1b in 0.37s\x1b\x1b =======================\x1b\n', stderr

## Full test suite

Runs all tests in `lib/tests/` (same as `cd lib && uv run pytest tests/ -v`).


In [8]:
run_pytest()

```text
[1m============================= test session starts ==============================[0m
platform linux -- Python 3.12.3, pytest-9.0.2, pluggy-1.6.0 -- /content/lib/.venv/bin/python
cachedir: .pytest_cache
rootdir: /content/lib
configfile: pyproject.toml
[1mcollecting ... [0mcollected 23 items

tests/test_commands.py::test_merge_overrides_copies_nested_files [32mPASSED[0m[32m  [  4%][0m
tests/test_commands.py::test_merge_overrides_skips_directories [32mPASSED[0m[32m    [  8%][0m
tests/test_commands.py::test_apply_copy_without_render_merges_patterns [32mPASSED[0m[32m [ 13%][0m
tests/test_commands.py::test_apply_copy_without_render_deduplicates [32mPASSED[0m[32m [ 17%][0m
tests/test_commands.py::test_apply_copy_without_render_non_list_existing_reset [32mPASSED[0m[32m [ 21%][0m
tests/test_commands.py::test_apply_copy_without_render_missing_ignore_noop [32mPASSED[0m[32m [ 26%][0m
tests/test_commands.py::test_apply_copy_without_render_empty_patterns_noop [32mPASSED[0m[32m [ 30%][0m
tests/test_commands.py::test_apply_copy_without_render_invalid_json_raises [32mPASSED[0m[32m [ 34%][0m
tests/test_commands.py::test_apply_copy_without_render_missing_cookiecutter_raises [32mPASSED[0m[32m [ 39%][0m
tests/test_commands.py::test_override_template_has_content_false_when_missing [32mPASSED[0m[32m [ 43%][0m
tests/test_commands.py::test_override_template_has_content_false_when_empty_dir [32mPASSED[0m[32m [ 47%][0m
tests/test_commands.py::test_override_template_has_content_true_with_file [32mPASSED[0m[32m [ 52%][0m
tests/test_commands.py::test_override_template_has_content_true_nested_file [32mPASSED[0m[32m [ 56%][0m
tests/test_commands.py::test_rich_exits_zero [32mPASSED[0m[32m                      [ 60%][0m
tests/test_commands.py::test_config_appends_bashrc_once [32mPASSED[0m[32m           [ 65%][0m
tests/test_commands.py::test_config_creates_bashrc [32mPASSED[0m[32m                [ 69%][0m
tests/test_commands.py::test_init_no_overrides_calls_cookiecutter_template_dir [32mPASSED[0m[32m [ 73%][0m
tests/test_commands.py::test_init_with_overrides_merges_and_calls_cookiecutter [32mPASSED[0m[32m [ 78%][0m
tests/test_commands.py::test_init_no_overrides_skips_merge_even_with_template_files [32mPASSED[0m[32m [ 82%][0m
tests/test_commands.py::test_hub_traefik_already_running [32mPASSED[0m[32m          [ 86%][0m
tests/test_commands.py::test_hub_creates_network_and_launches_traefik [32mPASSED[0m[32m [ 91%][0m
tests/test_commands.py::test_hub_aborts_when_network_declined [32mPASSED[0m[32m     [ 95%][0m
tests/test_commands.py::test_hub_aborts_when_traefik_launch_declined [32mPASSED[0m[32m [100%][0m

[32m============================== [32m[1m23 passed[0m[32m in 0.46s[0m[32m ==============================[0m
```

**Exit code:** `0`

CompletedProcess(args=['/content/lib/.venv/bin/python', '-m', 'pytest', 'tests/', '-v', '--tb=short'], returncode=0, stdout='\x1b============================= test session starts ==============================\x1b\nplatform linux -- Python 3.12.3, pytest-9.0.2, pluggy-1.6.0 -- /content/lib/.venv/bin/python\ncachedir: .pytest_cache\nrootdir: /content/lib\nconfigfile: pyproject.toml\n\x1bcollecting ... \x1bcollected 23 items\n\ntests/test_commands.py::test_merge_overrides_copies_nested_files \x1bPASSED\x1b\x1b  [  4%]\x1b\ntests/test_commands.py::test_merge_overrides_skips_directories \x1bPASSED\x1b\x1b    [  8%]\x1b\ntests/test_commands.py::test_apply_copy_without_render_merges_patterns \x1bPASSED\x1b\x1b [ 13%]\x1b\ntests/test_commands.py::test_apply_copy_without_render_deduplicates \x1bPASSED\x1b\x1b [ 17%]\x1b\ntests/test_commands.py::test_apply_copy_without_render_non_list_existing_reset \x1bPASSED\x1b\x1b [ 21%]\x1b\ntests/test_commands.py::test_apply_copy_without_render_missing_ig